# BigAlpha 2026 端到端模型推理

本 notebook 是唯一推理 notebook，平台调用 `main()`。模型训练由 `transformer_train.py` 完成，训练产物为 `transformer_model.json`。


In [ ]:
"""BigAlpha 端到端模型全局配置。提交要求的超参配置 + 随机种子集中于此。"""
import random

import numpy as np

# 字段清单依据 2026-07-27 平台探查结果 (bar15m 实际 schema, 盘口仅 5 档,
# 无 num_trades/avg_price/total_volume 系字段; 成交笔数为 deal_number)。
PRICE_COLS = ["open", "high", "low", "close", "pre_close",
              "bid_price1", "bid_price2", "bid_price3", "bid_price4", "bid_price5",
              "ask_price1", "ask_price2", "ask_price3", "ask_price4", "ask_price5"]
VOL_COLS = ["volume", "amount", "deal_number",
            "bid_volume1", "bid_volume2", "bid_volume3", "bid_volume4", "bid_volume5",
            "ask_volume1", "ask_volume2", "ask_volume3", "ask_volume4", "ask_volume5"]

CONFIG = {
    "table": "bigalpha_2026_stock_bar15m",
    "instruments_table": "bigalpha_2026_instruments",
    "exposure_table": "bigalpha_2026_exposure",
    "bars_per_day": 16,
    "price_cols": PRICE_COLS,
    "vol_cols": VOL_COLS,
    "feature_cols": PRICE_COLS + VOL_COLS,
    "min_time_coverage": 0.95,  # 每个股票日最少有效 (bar, 字段) 占比; 最终阈值由缺失率分布确定
    # 复权因子仅用于标签收益计算 (close*adjust_factor), 不作为模型输入字段
    "adjust_col": "adjust_factor",
    "lookback_days": 60,
    "infer_buffer_natural_days": 130,
    "chunk_size": 250,  # 每块股票数; 内存紧张(<32G)时降回 100
    "train_frac": 0.8,
    "seed": 42,
    "cache_path": "train_cache.npz",
    "checkpoint_path": "checkpoint.pt",
    "resume_checkpoint_path": "last_checkpoint.pt",
    # 官方提交的训练产物必须为 JSON；checkpoint.pt 仅用于本地断点/最佳权重。
    "artifact_path": "transformer_model.json",
    "label": {"mode": "residual", "horizon": 1, "winsor_pct": 1.0},
    "model": {"d_intra": 64, "heads_intra": 4, "layers_intra": 2, "ffn_intra": 128,
              "intra_chunk_size": 1024,
              "d_day": 128, "tau_cross": 8, "heads_cross": 2, "ffn_cross": 512,
              "use_cross_attn": True, "gate_init": -5.0, "glu_bottleneck": 64,
              "dropout": 0.1,
              "filter_kernel_sizes": [3, 7, 15]},
    "loss": {"mse_w": 0.1, "listnet_w": 0.05, "listnet_temp": 1.0},
    "train": {"epochs": 40, "lr": 5e-4, "weight_decay_head": 1.0, "clip": 1.0,
              "patience": 3, "swa_frac": 0.25, "stock_sample_ratio": 1.0,
              "log_every": 5, "checkpoint_every": 100,
              "time_budget_min": None,  # 由用户手动停止; Ctrl+C 会保存完整断点
              "metric_w": {"ic": 1.0, "icir": 0.02, "sharpe": 0.02}},
}

PARAM_MIN, PARAM_MAX = 100_000, 100_000_000


def validate_config(cfg):
    assert len(cfg["feature_cols"]) <= 100, "字段数超过赛规上限 100"
    assert len(cfg["feature_cols"]) == len(set(cfg["feature_cols"])), "字段重复"
    assert cfg["lookback_days"] <= 240, "回看窗口超过赛规上限 240 交易日"
    assert set(cfg["price_cols"]).isdisjoint(cfg["vol_cols"])
    assert cfg.get("resume_checkpoint_path", "last_checkpoint.pt") != \
        cfg["checkpoint_path"], "断点文件与 best model 必须使用不同路径"


def assert_param_count(n):
    assert PARAM_MIN <= n <= PARAM_MAX, \
        f"参数量 {n} 超出赛规区间 [{PARAM_MIN}, {PARAM_MAX}]"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def unify_missing(df, price_cols):
    """把哨兵值统一为 NaN，保留原始缺失位置的掩码语义。

    - inf / -inf → NaN
    - 非数值列 (空字符串等) → 强制转数值，失败变 NaN
    - price_cols 中负值 → NaN (价格不可能为负)
    """
    out = df.copy()
    out.replace([np.inf, -np.inf], np.nan, inplace=True)
    skip = {"date", "instrument"}
    for c in out.columns:
        if c in skip:
            continue
        if not pd.api.types.is_numeric_dtype(out[c]):
            out[c] = pd.to_numeric(out[c], errors="coerce")
    for c in price_cols:
        if c in out.columns:
            mask_neg = out[c] < 0
            if mask_neg.any():
                out.loc[mask_neg, c] = np.nan
    return out

In [ ]:
"""数据层：合规预处理、流式统计、按日透视、分块取数与缓存。"""
import numpy as np
import pandas as pd


def apply_field_transforms(df, cfg):
    """按字段统一变换（赛规允许类）：价格 log、量 log1p。"""
    out = df.copy()
    for c in cfg["price_cols"]:
        if c in out:
            out[c] = np.log(out[c].clip(lower=1e-6))
    for c in cfg["vol_cols"]:
        if c in out:
            out[c] = np.log1p(out[c].clip(lower=0))
    return out


class Normalizer:
    def __init__(self, mean, std):
        self.mean = np.asarray(mean, np.float32)
        self.std = np.asarray(std, np.float32)

    def apply(self, x):
        return ((x - self.mean) / self.std).astype(np.float32)


def pivot_to_days(df, feature_cols, bars_per_day, close_col="close",
                  min_time_coverage=0.95):
    """单只股票的 bar 级 df → (dates, X, mask, day_close)。"""
    n_feat = len(feature_cols)
    empty = (np.array([], "datetime64[D]"),
             np.zeros((0, bars_per_day, n_feat), np.float32),
             np.zeros((0, bars_per_day, n_feat), bool),
             np.array([], np.float64))
    if len(df) == 0:
        return empty
    feats = df[feature_cols].to_numpy(np.float32)
    closes_all = df[close_col].to_numpy(np.float64)
    day = df["date"].dt.normalize().to_numpy().astype("datetime64[D]")
    starts = np.flatnonzero(np.concatenate([[True], day[1:] != day[:-1]]))
    ends = np.append(starts[1:], len(day))
    counts = ends - starts

    if np.all(counts == bars_per_day) and np.isfinite(feats).all():
        return (day[starts],
                feats.reshape(-1, bars_per_day, n_feat),
                np.ones((len(starts), bars_per_day, n_feat), bool),
                closes_all[ends - 1])

    dates, arrs, masks, closes = [], [], [], []
    for s, e in zip(starts, ends):
        a = feats[s:e]
        n_bars = len(a)
        observed = np.isfinite(a)

        if observed.mean() < min_time_coverage:
            continue

        if n_bars >= bars_per_day:
            a = a[-bars_per_day:]
            observed = observed[-bars_per_day:]
        else:
            pad_n = bars_per_day - n_bars
            a = np.concatenate(
                [np.full((pad_n, n_feat), np.nan, dtype=np.float32), a])
            observed = np.concatenate(
                [np.zeros((pad_n, n_feat), dtype=bool), observed])

        dates.append(day[s])
        arrs.append(a)
        masks.append(observed)
        closes.append(closes_all[e - 1])
    if not dates:
        return empty
    return (np.array(dates), np.stack(arrs),
            np.stack(masks), np.array(closes, np.float64))


def iter_chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def list_instruments(query_fn, table, start, end):
    df = query_fn(f"SELECT DISTINCT instrument FROM {table}",
                  {"date": [start, end]})
    return sorted(df["instrument"].tolist())


def load_stock_arrays(query_fn, table, cfg, start, end, instruments):
    """分块查询 -> 缺失统一 -> 合规变换 -> 按日透视。"""
    import time as _time
    adj = cfg.get("adjust_col")
    extra = [adj] if adj and adj not in cfg["feature_cols"] else []
    cols = ", ".join(["date", "instrument"] + cfg["feature_cols"] + extra)
    sql = f"SELECT {cols} FROM {table} ORDER BY instrument, date"
    cols_no_adj = ", ".join(["date", "instrument"] + cfg["feature_cols"])
    sql_no_adj = f"SELECT {cols_no_adj} FROM {table} ORDER BY instrument, date"
    out = {}
    min_cov = cfg.get("min_time_coverage", 0.95)
    chunks = list(iter_chunks(instruments, cfg["chunk_size"]))
    t_start = _time.time()
    for ci, chunk in enumerate(chunks):
        t_c = _time.time()
        try:
            df = query_fn(sql, {"date": [start, end], "instrument": list(chunk)})
        except Exception:
            if extra:
                df = query_fn(sql_no_adj, {"date": [start, end], "instrument": list(chunk)})
                adj, extra, sql = None, [], sql_no_adj
            else:
                raise
        n_rows = 0 if df is None else len(df)
        elapsed = _time.time() - t_start
        eta = elapsed / (ci + 1) * (len(chunks) - ci - 1)
        print(f"[data] chunk {ci + 1}/{len(chunks)} rows={n_rows} "
              f"query={_time.time() - t_c:.0f}s elapsed={elapsed:.0f}s "
              f"eta={eta:.0f}s", flush=True)
        if df is None or len(df) == 0:
            continue
        t_p = _time.time()
        if adj and adj in df.columns:
            raw_close = df["close"].astype(np.float64) * df[adj].astype(np.float64)
        else:
            raw_close = df["close"].copy()
        df = unify_missing(df, cfg["price_cols"])
        df = apply_field_transforms(df, cfg)
        df["_raw_close"] = raw_close
        for ins, sub in df.groupby("instrument", sort=False):
            dates, X, mask, close = pivot_to_days(
                sub, cfg["feature_cols"], cfg["bars_per_day"],
                close_col="_raw_close", min_time_coverage=min_cov)
            if len(dates):
                out[str(ins)] = (dates, X.astype(np.float16),
                                 mask, close)
        print(f"[data] chunk {ci + 1}/{len(chunks)} processed "
              f"proc={_time.time() - t_p:.0f}s stocks_total={len(out)}", flush=True)
        del df
    return out

In [ ]:
"""模型：字段混合 -> 日内编码(ALiBi+global token) -> 日间GRU -> 逐时刻截面注意力(门控残差)
-> 末日query时序聚合 -> RankGLU打分头。"""
import torch
import torch.nn as nn
import torch.nn.functional as F


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def alibi_bias(n_heads, length, device):
    slopes = torch.tensor([2 ** (-8.0 * (i + 1) / n_heads) for i in range(n_heads)],
                          device=device)
    pos = torch.arange(length, device=device)
    dist = (pos[None, :] - pos[:, None]).abs().float()
    return -slopes[:, None, None] * dist[None]


class MultiScaleCausalFilter(nn.Module):
    """多尺度因果逐字段滤波 (Depthwise Causal Conv1D)。

    对每个字段 f 和每个尺度 k，计算 mask-aware 因果加权平均 L^{(k)} 作为低频分量，
    通过可学习 gate 控制平滑/锐化程度:

      out = x + ∑_k tanh(g_k) · (L^{(k)} - x)

    - tanh(g) ∈ (-1, 1): 正=向平滑移动 (低通), 负=向锐化移动 (高通), 零=恒等
    - 初始化 g=0 → tanh(0)=0 → 滤波器起始为精确恒等变换
    - 逐字段 (groups=n_feat): 不同字段在这一层不混合
    - 因果 (left-pad k-1): t 时刻只使用 ≤t 的数据
    - mask-aware: 仅对真实观测加权, 缺失位置贡献 0
    - softmax 权重: w ≥ 0, ∑w = 1 → 可解释为平滑滤波器
    """

    def __init__(self, n_feat, kernel_sizes=(3, 7, 15), eps=1e-5):
        super().__init__()
        self.n_feat = n_feat
        self.kernel_sizes = tuple(kernel_sizes)
        self.n_scales = len(kernel_sizes)
        self.max_k = max(kernel_sizes)
        self.eps = eps

        # 逐字段平滑核权重 → softmax → 非负、和为 1
        # 初始化为 0 → softmax 接近均匀分布 → 近似简单移动平均
        self.logits = nn.Parameter(torch.zeros(n_feat, self.max_k))

        # 逐字段逐尺度的 gate: tanh(g) ∈ (-1, 1) 控制平滑/锐化
        # 初始化为 0 → tanh(0)=0 → 精确恒等, 训练中逐步分化
        self.gate_raw = nn.Parameter(torch.zeros(n_feat, self.n_scales))

    def _causal_weighted_avg(self, x, mask, w, k):
        N, B, F_dim = x.shape
        x_pad = F.pad(x, (0, 0, k - 1, 0))
        m_pad = F.pad(mask.float(), (0, 0, k - 1, 0))
        x_win = x_pad.unfold(1, k, 1)
        m_win = m_pad.unfold(1, k, 1)
        w_ = w.unsqueeze(0).unsqueeze(0)
        num = (x_win * w_ * m_win).sum(dim=-1)
        den = (m_win * w_).sum(dim=-1).clamp(min=self.eps)
        return num / den

    def forward(self, x, observed_mask=None):
        if observed_mask is None:
            observed_mask = torch.ones_like(x, dtype=torch.bool)
        w_all = F.softmax(self.logits, dim=-1)
        gate = torch.tanh(self.gate_raw)

        out = x

        for si, k in enumerate(self.kernel_sizes):
            wk = w_all[:, :k]
            low = self._causal_weighted_avg(x, observed_mask, wk, k)
            g = gate[:, si].view(1, 1, -1)

            # g > 0 → 向平滑方向移动 (低通滤波)
            # g < 0 → 向锐化方向移动 (高通增强)
            # g = 0 → 精确恒等
            out = out + g * (low - x)

        return out


class FieldMix(nn.Module):
    """字段维混合块 (StockMixer): 端到端学跨字段交互, 替代被禁止的人工跨字段算子。"""

    def __init__(self, n_feat):
        super().__init__()
        self.norm = nn.LayerNorm(n_feat)
        self.fc1 = nn.Linear(n_feat, n_feat)
        self.fc2 = nn.Linear(n_feat, n_feat)
        self.act = nn.Hardswish()

    def forward(self, x):
        return x + self.fc2(self.act(self.fc1(self.norm(x))))


class _EncoderLayer(nn.Module):
    def __init__(self, d, heads, ffn, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, ffn), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(ffn, d))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask):
        h = self.n1(x)
        a, _ = self.attn(h, h, h, attn_mask=mask, need_weights=False)
        x = x + self.drop(a)
        return x + self.drop(self.ffn(self.n2(x)))


class IntradayEncoder(nn.Module):
    """日内 bar 序列 -> 当日向量。ALiBi 距离衰减偏置 (TIPS) + global token 聚合 (TimeXer)。"""

    def __init__(self, n_feat, d, heads, layers, ffn, bars, dropout):
        super().__init__()
        self.proj = nn.Linear(n_feat, d)
        self.pos = nn.Parameter(torch.zeros(1, bars, d))
        self.glb = nn.Parameter(torch.zeros(1, 1, d))
        self.layers = nn.ModuleList(_EncoderLayer(d, heads, ffn, dropout)
                                    for _ in range(layers))
        self.heads = heads
        self.out_norm = nn.LayerNorm(d)
        bias = torch.zeros(heads, bars + 1, bars + 1)
        bias[:, 1:, 1:] = alibi_bias(heads, bars, "cpu")
        self.register_buffer("attn_bias", bias, persistent=False)

    def forward(self, x):  # (B*, bars, F) -> (B*, d)
        h = self.proj(x) + self.pos
        h = torch.cat([self.glb.expand(h.shape[0], -1, -1), h], dim=1)
        mask = self.attn_bias.repeat(h.shape[0], 1, 1)
        for lyr in self.layers:
            h = lyr(h, mask)
        return self.out_norm(h[:, 0])


class InterDayEncoder(nn.Module):
    def __init__(self, d_in, d):
        super().__init__()
        self.gru = nn.GRU(d_in, d, batch_first=True)

    def forward(self, x):  # (N, L, d_in) -> (N, L, d)
        out, _ = self.gru(x)
        return out


class CrossSectionBlock(nn.Module):
    """截面注意力: batch 维=时间位置, 序列维=股票; 无位置编码 (排列不变, iTransformer);
    门控残差近零初始化 (WaveLSFormer): 先学单股基线, 训练中逐步启用截面信息。"""

    def __init__(self, d, heads, ffn, dropout, gate_init):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
        self.gate_attn = nn.Parameter(torch.tensor(float(gate_init)))
        self.ffn_norm = nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, ffn), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(ffn, d))
        self.gate_ffn = nn.Parameter(torch.tensor(float(gate_init)))

    def forward(self, h):  # (tau, N, d) -> (tau, N, d)
        x = self.norm(h)
        a, _ = self.attn(x, x, x, need_weights=False)
        h = h + torch.sigmoid(self.gate_attn) * a
        return h + torch.sigmoid(self.gate_ffn) * self.ffn(self.ffn_norm(h))


class TemporalAggregator(nn.Module):
    """以最新一天为 query 的时序注意力加权 (MASTER/DTML)。"""

    def __init__(self, d):
        super().__init__()
        self.q, self.k = nn.Linear(d, d), nn.Linear(d, d)
        self.scale = d ** -0.5

    def forward(self, h):  # (N, tau, d) -> (N, d)
        w = torch.softmax((self.q(h[:, -1:]) @ self.k(h).transpose(1, 2)) * self.scale, -1)
        return (w @ h).squeeze(1)


class RankGLUHead(nn.Module):
    """线性直通 + γ·瓶颈 GLU 门控残差 (RankGLU): 稳定排序几何 + 有界低秩非线性。"""

    def __init__(self, d, b, gamma=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.lin = nn.Linear(d, 1)
        self.v, self.g = nn.Linear(d, b), nn.Linear(d, b)
        self.out = nn.Linear(b, 1)
        self.gamma = nn.Parameter(torch.tensor(float(gamma)))

    def forward(self, e):
        e = self.norm(e)
        z = self.v(e) * torch.sigmoid(self.g(e))
        return (self.lin(e) + self.gamma * self.out(z)).squeeze(-1)


class StockScorer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        m = cfg["model"]
        n_feat = len(cfg["feature_cols"])
        self.tau = m["tau_cross"]
        self.use_cross = m["use_cross_attn"]
        self.intra_chunk_size = m.get("intra_chunk_size", 4096)
        k_sizes = m.get("filter_kernel_sizes")
        self.filter = MultiScaleCausalFilter(n_feat, k_sizes) if k_sizes else None
        self.fieldmix = FieldMix(n_feat)
        self.intra = IntradayEncoder(n_feat, m["d_intra"], m["heads_intra"],
                                     m["layers_intra"], m["ffn_intra"],
                                     cfg["bars_per_day"], m["dropout"])
        self.inter = InterDayEncoder(m["d_intra"], m["d_day"])
        self.cross = CrossSectionBlock(m["d_day"], m["heads_cross"], m["ffn_cross"],
                                       m["dropout"], m["gate_init"])
        self.agg = TemporalAggregator(m["d_day"])
        self.head = RankGLUHead(m["d_day"], m["glu_bottleneck"])

    def _intra_block(self, part, part_mask):
        """Filter + FieldMix + IntradayEncoder."""
        if self.filter is not None and part_mask is not None:
            part = self.filter(part, part_mask)
        return self.intra(self.fieldmix(part))

    def encode_days(self, x, observed_mask=None):  # (N, L, B, F) -> (N, L, d_day)
        N, L, B, F_dim = x.shape
        flat = x.reshape(N * L, B, F_dim)
        if observed_mask is not None:
            flat_mask = observed_mask.reshape(N * L, B, F_dim)
        else:
            flat_mask = None
        chunks = []
        for start in range(0, len(flat), self.intra_chunk_size):
            part = flat[start:start + self.intra_chunk_size]
            part_mask = (flat_mask[start:start + self.intra_chunk_size]
                         if flat_mask is not None else None)
            chunks.append(self._intra_block(part, part_mask))
        v = torch.cat(chunks, dim=0).reshape(N, L, -1)
        return self.inter(v)

    def score_day(self, H):  # (N, tau, d_day) -> (N,)
        if self.use_cross:
            H = self.cross(H.transpose(0, 1)).transpose(0, 1)
        return self.head(self.agg(H))

    def forward(self, x, observed_mask=None):
        H = self.encode_days(x, observed_mask)
        return self.score_day(H[:, -self.tau:])

In [ ]:
"""JSON 训练产物的编解码：把 PyTorch state_dict 无损编码为官方要求的 JSON 格式。"""
import base64
import json
import os

import numpy as np
import torch


def export_model_json(ckpt, path):
    """把 PyTorch checkpoint 无损编码为官方要求的 JSON 训练产物。"""
    encoded_state = {}
    for name, value in ckpt["state_dict"].items():
        array = value.detach().cpu().contiguous().numpy()
        encoded_state[name] = {
            "dtype": array.dtype.str,
            "shape": list(array.shape),
            "data_b64": base64.b64encode(array.tobytes()).decode("ascii"),
        }
    cfg = ckpt["config"]
    field_order = cfg["feature_cols"]
    transform_methods = {}
    for c in cfg["price_cols"]:
        if c in field_order:
            transform_methods[c] = "log"
    for c in cfg["vol_cols"]:
        if c in field_order:
            transform_methods[c] = "log1p"
    artifact = {
        "format_version": 1,
        "artifact_type": "bigalpha_pytorch_state_dict",
        "state_dict": encoded_state,
        "field_order": field_order,
        "transform": transform_methods,
        "mean": np.asarray(ckpt["mean"], np.float32).tolist(),
        "std": np.asarray(ckpt["std"], np.float32).tolist(),
        "epsilon": 1e-6,
        "fit_date_range": ckpt.get("fit_date_range", []),
        "config": cfg,
        "best_epoch": ckpt.get("best_epoch"),
        "val_metrics": ckpt.get("val_metrics"),
        "metric_version": ckpt.get("metric_version", 1),
    }
    target = os.fspath(path)
    temporary = target + ".tmp"
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(artifact, handle, ensure_ascii=False, separators=(",", ":"))
    os.replace(temporary, target)
    print(f"[artifact] saved {target}", flush=True)
    return target


def load_model_json(path):
    """读取 JSON 训练产物并还原为 run_inference 使用的 checkpoint。

    向后兼容: 旧版产物缺少 field_order/transform/fit_date_range/epsilon 时
    不报错，从 config 推导 field_order 和 transform。"""
    with open(path, "r", encoding="utf-8") as handle:
        artifact = json.load(handle)
    if artifact.get("format_version") != 1:
        raise ValueError(
            f"不支持的模型产物版本: {artifact.get('format_version')}"
        )
    state_dict = {}
    for name, item in artifact["state_dict"].items():
        raw = base64.b64decode(item["data_b64"])
        array = np.frombuffer(raw, dtype=np.dtype(item["dtype"])).copy()
        array = array.reshape(item["shape"])
        state_dict[name] = torch.from_numpy(array)
    return {
        "state_dict": state_dict,
        "mean": np.asarray(artifact["mean"], np.float32),
        "std": np.asarray(artifact["std"], np.float32),
        "config": artifact["config"],
        "best_epoch": artifact.get("best_epoch"),
        "val_metrics": artifact.get("val_metrics"),
        "metric_version": artifact.get("metric_version", 1),
        "field_order": artifact.get("field_order",
                                    artifact["config"]["feature_cols"]),
        "transform": artifact.get("transform"),
        "fit_date_range": artifact.get("fit_date_range", []),
        "epsilon": artifact.get("epsilon", 1e-6),
    }

In [ ]:
"""推理：分块编码 -> 按日截面打分 -> instruments 骨架 left join。

构造上保证平台三项校验必过: 三列名精确匹配、评估区间交易日完整、逐日缺失率为 0
（无法计算的 (date, instrument) 填截面中性值 0）。
"""
import numpy as np
import pandas as pd
import torch



def run_inference(ckpt, query_fn, infer_table, instruments_query_fn,
                  start_date, end_date, device=None):
    cfg = ckpt["config"]
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = StockScorer(cfg).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    if not cfg.get("skip_param_check"):
        assert_param_count(count_params(model))
    normalizer = Normalizer(ckpt["mean"], ckpt["std"])

    skeleton = instruments_query_fn(start_date, end_date)[["date", "instrument"]].copy()
    skeleton["date"] = pd.to_datetime(skeleton["date"]).dt.normalize()
    instruments = sorted(skeleton["instrument"].unique().tolist())

    buf = (pd.to_datetime(start_date)
           - pd.Timedelta(days=cfg["infer_buffer_natural_days"])).strftime("%Y-%m-%d 00:00:00")
    sd = pd.to_datetime(start_date).normalize()
    ed = pd.to_datetime(end_date).normalize()

    tau = cfg["model"]["tau_cross"]
    per_day = {}  # date -> list[(instrument, H_tau (tau, d_day))]
    with torch.no_grad():
        for chunk in iter_chunks(instruments, cfg["chunk_size"]):
            arrays = load_stock_arrays(query_fn, infer_table, cfg,
                                       buf, str(end_date), list(chunk))
            for ins, (dates, X, mask, _) in arrays.items():
                idx = [i for i, d in enumerate(dates)
                       if sd <= pd.Timestamp(d) <= ed]
                if not idx:
                    continue
                Xf = normalizer.apply(X.astype(np.float32))
                Xf = np.where(np.isfinite(Xf), Xf, 0.0)  # 缺失→标准化后均值
                Xn = torch.from_numpy(Xf)[None].to(device)       # (1, D, B, F)
                Mn = torch.from_numpy(mask)[None].to(device)     # (1, D, B, F)
                H = model.encode_days(Xn, observed_mask=Mn)[0].cpu()  # (D, d_day)
                for i in idx:
                    h = H[max(0, i + 1 - tau): i + 1]
                    if len(h) < tau:  # 历史不足: 重复首日补齐
                        h = torch.cat([h[:1].expand(tau - len(h), -1), h])
                    per_day.setdefault(pd.Timestamp(dates[i]), []).append((ins, h))
            del arrays

        rows = []
        for d, items in per_day.items():
            Ht = torch.stack([h for _, h in items]).to(device)  # (N, tau, d_day)
            scores = model.score_day(Ht).cpu().numpy()
            rows += [(d, ins, float(s)) for (ins, _), s in zip(items, scores)]

    pred = pd.DataFrame(rows, columns=["date", "instrument", "score"])
    out = skeleton.merge(pred, on=["date", "instrument"], how="left")
    out["score"] = out["score"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return out[["date", "instrument", "score"]].reset_index(drop=True)

In [ ]:
def main(datasources, start_date, end_date):
    """平台评判入口：加载 JSON 训练产物，对注入的测试表推理。"""
    import dai
    import structlog

    logger = structlog.get_logger()
    infer_table = datasources["bar15m"]

    def query_fn(sql, filters):
        return dai.query(sql, filters=filters, compression=True).df()

    def instruments_query_fn(sd, ed):
        return dai.query(
            f"SELECT date, instrument FROM {CONFIG['instruments_table']}",
            filters={"date": [sd, ed]},
            compression=True,
        ).df()

    ckpt = load_model_json(CONFIG["artifact_path"])
    required = {"state_dict", "mean", "std", "config"}
    missing = required.difference(ckpt)
    if missing:
        raise KeyError(f"checkpoint 缺少字段: {sorted(missing)}")
    logger.info(
        "JSON 模型加载完成",
        best_epoch=ckpt.get("best_epoch"),
        val_metrics=ckpt.get("val_metrics"),
    )
    result = run_inference(
        ckpt,
        query_fn,
        infer_table,
        instruments_query_fn,
        start_date,
        end_date,
    )
    logger.info(
        "分数构建完成",
        rows=len(result),
        days=result["date"].nunique(),
        instruments=result["instrument"].nunique(),
    )
    return result


def local_evaluate(
    start_date="2023-10-01 00:00:00",
    end_date="2023-12-31 23:59:59",
):
    """仅供开发环境手动调用；平台评判不会调用此函数。"""
    from bigmodule import M

    datasources = {"bar15m": "bigalpha_2026_stock_bar15m"}
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())
    return M.bigalpha_eval._latest(factor_data=score_data, show=True)

In [ ]:
# 本地调试时手动取消下一行注释；正式提交保持注释。
# local_result = local_evaluate()
